# MNE: MEG/EEG Analysis

Load the MNE sample dataset (auditory + visual evoked responses), preprocess, compute ERPs and source localization on the fsaverage surface.

## Prerequisites

- **Python 3.9+**
- Basic familiarity with EEG/MEG signal processing concepts (epochs, ERPs, time-frequency analysis)
- Understanding of inverse problems and source localization is helpful but not required

## Setup

This notebook uses the MNE sample dataset, which will be automatically downloaded (~1.5 GB) on first run.
The data contains simultaneous MEG and EEG recordings from an auditory and visual stimulus paradigm.

In [ ]:
# Install required packages
!pip install mne matplotlib numpy

In [ ]:
import mne
import os
import matplotlib.pyplot as plt
import numpy as np

# Download the MNE sample dataset (uses default MNE data directory)
data_path = mne.datasets.sample.data_path()
raw_fname = data_path / 'MEG' / 'sample' / 'sample_audvis_filt-0-40_raw.fif'

# Load raw MEG/EEG recording
raw = mne.io.read_raw_fif(raw_fname, preload=True)
print(raw.info)
print(f'Duration: {raw.times[-1]:.1f}s | Channels: {raw.info["nchan"]} | Sfreq: {raw.info["sfreq"]}Hz')

In [ ]:
# Find events (auditory L/R, visual L/R)
events = mne.find_events(raw, stim_channel='STI 014')
event_id = {'auditory/left': 1, 'auditory/right': 2, 'visual/left': 3, 'visual/right': 4}
mne.viz.plot_events(events, event_id=event_id, sfreq=raw.info['sfreq'])
plt.show()

In [ ]:
# Epoch around events and compute ERPs
picks = mne.pick_types(raw.info, meg=True, eeg=True, eog=False)
epochs = mne.Epochs(raw, events, event_id, tmin=-0.2, tmax=0.5,
                    picks=picks, baseline=(None, 0), preload=True)
print(epochs)

# Evoked (ERP/ERF)
evoked_aud = epochs['auditory/left'].average()
evoked_vis = epochs['visual/left'].average()
mne.viz.plot_compare_evokeds({'auditory': evoked_aud, 'visual': evoked_vis},
                             picks='eeg', title='ERP: Auditory vs Visual')
plt.show()

In [ ]:
# Time-frequency decomposition (induced power)
from mne.time_frequency import tfr_morlet

# Start at 8 Hz: wavelets at 4 Hz need ~119 samples but epochs are only 106 samples wide.
# At 8 Hz with n_cycles=4, wavelet length fits within the 106-sample epoch window.
freqs = np.logspace(np.log10(8), np.log10(40), 15)
n_cycles = np.minimum(freqs / 4.0, 4.0)  # cap cycles so no wavelet exceeds epoch length

power = tfr_morlet(epochs['auditory/left'], freqs=freqs, n_cycles=n_cycles,
                   return_itc=False, average=True)
power.plot([0], baseline=(-0.2, 0), mode='logratio', title='Auditory TFR \u2014 Morlet wavelet')
plt.show()

In [ ]:
# Source localization \u2014 use the 'sample' subject (bundled with MNE sample dataset)
subjects_dir = data_path / 'subjects'
fname_fwd = data_path / 'MEG' / 'sample' / 'sample_audvis-meg-eeg-oct-6-fwd.fif'
fwd = mne.read_forward_solution(fname_fwd)
noise_cov = mne.compute_covariance(epochs, tmax=0.0)
inv = mne.minimum_norm.make_inverse_operator(evoked_aud.info, fwd, noise_cov)
stc = mne.minimum_norm.apply_inverse(evoked_aud, inv, lambda2=1./9., method='dSPM')
print('Source estimate shape:', stc.data.shape, '| subject:', stc.subject)

# matplotlib backend: views must use 3-letter abbreviations ('lat' not 'lateral')
brain = stc.plot(subjects_dir=str(subjects_dir), subject='sample',
                 backend='matplotlib',
                 hemi='lh', views='lat',
                 initial_time=0.1, title='Auditory ERP \u2014 dSPM source')
plt.show()

## References

- Gramfort, A., Luessi, M., Larson, E., Engemann, D. A., Strohmeier, D., Brodbeck, C., ... & H\u00e4m\u00e4l\u00e4inen, M. S. (2013). MEG and EEG data analysis with MNE-Python. *Frontiers in Neuroscience*, 7, 267. https://doi.org/10.3389/fnins.2013.00267
- Gramfort, A., Luessi, M., Larson, E., Engemann, D. A., Strohmeier, D., Brodbeck, C., ... & H\u00e4m\u00e4l\u00e4inen, M. S. (2014). MNE software for processing MEG and EEG data. *NeuroImage*, 86, 446-460. https://doi.org/10.1016/j.neuroimage.2013.10.027
- Dale, A. M., Liu, A. K., Fischl, B. R., Buckner, R. L., Belliveau, J. W., Lewine, J. D., & Halgren, E. (2000). Dynamic statistical parametric mapping: combining fMRI and MEG for high-resolution imaging of cortical activity. *Neuron*, 26(1), 55-67. https://doi.org/10.1016/S0896-6273(00)81138-1
- MNE-Python documentation: https://mne.tools/stable/